In [ ]:
# import packages
import fiftyone as fo
from tqdm import tqdm

In [ ]:
# load data from images_v4 to be visualized 
from db_utils import read_dataset_from_table
df = read_dataset_from_table(table_name='images_v4', 
                             dataset_name='Trail Camera Images of New Zealand Animals', 
                             cloud_proxy_path='/home/yijin/sentinel/sentinel-data-pipelines-v4/cloud_proxy.sh')


In [ ]:
# metadata fields to include 
image_metadata_keys = ['file_name', 'image_id', 'cloud_path', 'dataset_name', 'datetime', 'latitude', 'longitude', 
                       'country_code', 'country_name', 'location_id', 'camera_id', 'seq_id', 'frame_num',
                       'phash', 'flags', 'detector_algorithm', 'host_location', 'camera_trap', 'rights_holder']
bbox_metadata_keys = ['bb_id','bb_confidence', 'original_label','common_name',
                      'bb_confirmed','label_confirmed', 'wrong_label', 'RDE_done',
                      'kingdom', 'phylum', 'class','order', 'family', 'genus', 'species', 'subspecies', 
                      'individual_id', 'sex', 'behavior', 'lifeStage', 'feature', 'color']

# process metadata image-by-image
samples = []
grouped = df.groupby("cloud_path")

for cloud_path, group in tqdm(grouped, desc=f'Processing {len(grouped)} images'):
    # retrieve image metadata from the first row (should be consistent across rows)
    row = group.iloc[0] 
    sample = fo.Sample(filepath=cloud_path)
    for key in image_metadata_keys:
        sample[key] = row[key]

    # pre-extract bounding box coordinates (need to convert them to fiftyone format later)
    xmins = group["voc_xmin"].astype(float).values
    ymins = group["voc_ymin"].astype(float).values
    xmaxs = group["voc_xmax"].astype(float).values
    ymaxs = group["voc_ymax"].astype(float).values

    # pre-extract all bbox metadata columns as arrays
    bbox_cols = {key: group[key].values for key in bbox_metadata_keys}

    # construct detections for this image
    detections = []
    for i in range(len(group)):
        x1, y1, x2, y2 = xmins[i], ymins[i], xmaxs[i], ymaxs[i]
        det_kwargs = {key: bbox_cols[key][i] for key in bbox_metadata_keys} # extract metadata for this annotation
        det_kwargs["bounding_box"] = [x1, y1, x2 - x1, y2 - y1]
        det_kwargs["label"] = det_kwargs.pop("common_name") # set box labels to common name
        det_kwargs["confidence"] = det_kwargs.pop("bb_confidence") # confidence is a built-in field that allows for downstream filtering in fiftyone
        detections.append(fo.Detection(**det_kwargs))

    sample["annotations"] = fo.Detections(detections=detections)
    samples.append(sample)

In [ ]:
# add_samples to dataset
dataset = fo.Dataset("nz_trailcams", persistent=True) # stored at ~/.fiftyone/var/lib/mongo
dataset.add_samples(samples)

In [ ]:
session = fo.launch_app(dataset)

In [ ]:
session

In [ ]:
# # close everything
# session.close()
# fo.close_app()